# CS781 Phase 3. Encoder-decoder LoRA preflight

This notebook checks the model, sentence alignment and GPU setup before the full [Encoder-decoder LoRA training run](phase3_encoder_decoder_lora.ipynb). The learning steps are temporary and the preflight exports a proof for the training run.

The saved preflight passed on two T4 GPUs and selected a chunk microbatch of 16 for the subsequent training notebook.


## 1. Inputs and environment

1. Start a Kaggle session with two T4 GPUs. Use the session's PyTorch installation.
2. Attach `prep_002_mixed16k` with its tokenizer and manifest. Attach the original
   [Phase 1 run](../phase1/phase1_encoder_decoder_training.ipynb) `encdec_002_mixed217m_stable` with `best_model.pt`
   and its source files. Set `PREP_INPUT` and `ENCDEC_INPUT` if automatic
   discovery is ambiguous.
3. Set `RR_DATA_INPUT` to the official data folder if it is already available.
   For the pinned download, enable Internet and provide the Kaggle secret `HF_TOKEN`.


In [1]:
%pip install -q huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import random
import shutil
import sys
import time
import traceback

import numpy as np
import pandas as pd
import torch
from IPython.display import display

RUN_ID='encdec_rr_lora_preflight_001'
PREP_INPUT=None
ENCDEC_INPUT=None
RR_DATA_INPUT=None
ROOT=Path('/kaggle/working/cs781_phase3')
SRC=Path('/kaggle/working/cs781_phase3_encdec_preflight_src')
RUN_DIR=ROOT/'preflights'/RUN_ID
for folder in (ROOT,SRC,RUN_DIR):
    folder.mkdir(parents=True,exist_ok=True)

SETTINGS={
    'seed':42,'window_sentences':128,'overlap_sentences':32,
    'chunk_tokens':256,'chunk_microbatch':8,'global_window_batch':2,'accumulation_steps':2,
    'gradient_checkpointing':True,'projection_dim':256,'context_hidden':128,'head_dropout':0.20,
    'lora_rank':32,'lora_alpha':64,'lora_dropout':0.05,'lora_lr':2e-4,
    'head_lr':3e-4,'crf_lr':1e-3,'weight_decay':0.01,'gradient_clip':1.0,
    'class_weight_cap':5.0,'aux_ce_coefficient':0.5,'require_two_gpus':True,
}
PROPOSED_TRAINING={'epochs':35,'minimum_epochs':15,'patience':8,'early_stop_min_delta':1e-4,
                  'warmup_fraction':0.05,'minimum_lr_fraction':0.01,
                  'train_metrics_every_epochs':3,'dev_metrics_every_epochs':1}
LEARNING_STEPS=40
torch.set_num_threads(2)
random.seed(SETTINGS['seed'])
np.random.seed(SETTINGS['seed'])
torch.manual_seed(SETTINGS['seed'])
if not torch.cuda.is_available() or torch.cuda.device_count()<2:
    raise RuntimeError('Select Kaggle T4 x2 for this preflight.')
if any(torch.cuda.memory_allocated(i)>256*2**20 for i in range(2)):
    raise RuntimeError('Restart the kernel first: another model is still allocated on a GPU.')
torch.cuda.manual_seed_all(SETTINGS['seed'])
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(2)])
print('Results:',RUN_DIR)
print('This notebook runs bounded probes only; the 35-epoch training loop is not included.')

GPUs: ['Tesla T4', 'Tesla T4']
Results: /kaggle/working/cs781_phase3/preflights/encdec_rr_lora_preflight_001
This notebook runs bounded probes only; the 35-epoch training loop is not included.


## 2. Preflight implementation

Write the same backbone, tokenizer and adaptation code as the training notebook, with helpers for numerical, learning and resource checks.


In [3]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/phase1_utils.py
"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/phase1_utils.py


In [4]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/a1_tokenizer.py
"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/a1_tokenizer.py


In [5]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/architecture_models.py
"""Three from-scratch Phase 1 Transformers; course blocks with cited extensions.

SDPA Boolean masks use True for ALLOWED connections. All blocks are pre-norm.
Training uses full teacher-forced sequences. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "encoder_decoder"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    encoder_layers: int = 8
    decoder_layers: int = 6
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128


def configuration(architecture):
    if architecture not in ("encoder_decoder", "prefix", "efficient"):
        raise ValueError("Unknown architecture")
    return ModelConfig(architecture=architecture, d_ff=4864 if architecture == "efficient" else 4096)


def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


def valid_keys(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, x, allowed):
        x = x + self.attn(self.ln1(x), allowed=allowed)
        return x + self.ffn(self.ln2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2, self.ln3 = (nn.LayerNorm(cfg.d_model) for _ in range(3))
        self.self_attn, self.cross_attn, self.ffn = Attention(cfg), Attention(cfg), FeedForward(cfg)

    def forward(self, x, memory, allowed):
        x = x + self.self_attn(self.ln1(x), causal=True)
        x = x + self.cross_attn(self.ln2(x), memory=memory, allowed=allowed)
        return x + self.ffn(self.ln3(x))


class PrefixBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, prompt, decoder, prefix_allowed):
        # The prompt cannot see decoder tokens; decoder queries see both parts.
        p = self.ln1(prompt)
        d = self.ln1(decoder)
        pk, pv = self.attn.keys_values(p)
        dk, dv = self.attn.keys_values(d)
        prompt = prompt + self.attn.attend(self.attn.query(p), pk, pv, prefix_allowed)
        t = decoder.shape[1]
        triangular = torch.ones(t, t, dtype=torch.bool, device=decoder.device).tril()
        allowed = torch.cat([prefix_allowed.expand(-1, -1, t, -1),
                             triangular[None, None].expand(decoder.shape[0], 1, -1, -1)], dim=-1)
        decoder = decoder + self.attn.attend(self.attn.query(d), torch.cat([pk, dk], dim=2),
                                            torch.cat([pv, dv], dim=2), allowed)
        return prompt + self.ffn(self.ln2(prompt)), decoder + self.ffn(self.ln2(decoder))


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None, role=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        if role is not None:
            x = x + self.segment_emb.weight[role]
        return x


class EncoderDecoderLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.encoder_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.decoder_layers)])
        self.encoder_norm, self.decoder_norm = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.initialize()

    def encode(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt)
        for block in self.encoder:
            x = block(x, allowed)
        return self.encoder_norm(x), allowed

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        memory, allowed = self.encode(prompt, lengths)
        x = self.embed(decoder_ids)
        for block in self.decoder:
            x = block(x, memory, allowed)
        return self.head(self.decoder_norm(x))

    def start_generation(self, prompt, lengths):
        memory, allowed = self.encode(prompt, lengths)
        layers = []
        for block in self.decoder:
            k, v = block.cross_attn.keys_values(memory)
            layers.append({"cross_k": k, "cross_v": v,
                           "k": k[:, :, :0], "v": v[:, :, :0]})
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        position = cache["decoder_position"]
        x = self.embed(ids, offset=torch.full_like(cache["lengths"], position))
        for block, layer in zip(self.decoder, cache["layers"]):
            h = block.ln1(x)
            k, v = block.self_attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            # One new query may see all cached decoder keys, including itself.
            x = x + block.self_attn.attend(block.self_attn.query(h), layer["k"], layer["v"])
            x = x + block.cross_attn.attend(block.cross_attn.query(block.ln2(x)),
                                          layer["cross_k"], layer["cross_v"], cache["allowed"])
            x = x + block.ffn(block.ln3(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.decoder_norm(x))[:, -1], cache


class PrefixLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.segment_emb = nn.Embedding(2, cfg.d_model)
        self.blocks = nn.ModuleList([PrefixBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        p, d = self.embed(prompt, role=0), self.embed(decoder_ids, offset=lengths, role=1)
        for block in self.blocks:
            p, d = block(p, d, allowed)
        return self.head(self.final_norm(d))

    def start_generation(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt, role=0)
        layers = []
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layers.append({"prefix_k": k, "prefix_v": v, "k": k[:, :, :0], "v": v[:, :, :0]})
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"], role=1)
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            allowed = torch.cat([cache["allowed"],
                torch.ones((ids.shape[0], 1, 1, layer["k"].shape[2]), device=ids.device, dtype=torch.bool)], dim=-1)
            x = x + block.attn.attend(block.attn.query(h),
                torch.cat([layer["prefix_k"], layer["k"]], dim=2),
                torch.cat([layer["prefix_v"], layer["v"]], dim=2), allowed)
            x = x + block.ffn(block.ln2(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.final_norm(x))[:, -1], cache


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def build_model(cfg):
    constructors = {"encoder_decoder": EncoderDecoderLM, "prefix": PrefixLM, "efficient": EfficientLM}
    return constructors[cfg.architecture](cfg)


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


def logits_and_labels(lm, x, y, prefix_length):
    if lm.cfg.architecture == "efficient":
        return lm(x), y
    p = int(prefix_length)
    if not 0 < p < x.shape[1]:
        raise ValueError("Need a nonempty prefix and continuation")
    labels = x[:, p:]
    bos = torch.full_like(labels[:, :1], 2)
    decoder = torch.cat([bos, labels[:, :-1]], dim=1)
    return lm(x[:, :p], decoder), labels


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        prefix_length = int(prefixes[0].item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, prefix_length)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/architecture_models.py


In [6]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/rr_data.py
import hashlib
import json
import math
import pickle
import re
import shutil
import time
import zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
ROOT = Path('/kaggle/working/cs781_phase3')
INPUT_ROOTS = [Path('/kaggle/input')]

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(2**20), b''):
            h.update(part)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def read_json(path):
    def unique_pairs(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'Duplicate JSON key: {key!r}')
            result[key] = value
        return result
    def reject_constant(value):
        raise ValueError(f'Invalid JSON constant: {value}')
    return json.loads(Path(path).read_text(encoding='utf-8'),
                      object_pairs_hook=unique_pairs, parse_constant=reject_constant)

def find_input(filename, supplied=None, hint=None, required=True):
    archives = []
    if supplied is not None:
        p = Path(supplied)
        if p.is_file() and p.suffix.lower() == '.zip':
            candidates, archives = [], [p]
        else:
            candidates = [p] if p.is_file() and p.name == filename else list(p.rglob(filename))
            archives = list(p.rglob('*.zip')) if p.is_dir() else []
    else:
        candidates = [p for root in INPUT_ROOTS if root.exists() for p in root.rglob(filename)]
        if hint is not None:
            candidates = [p for p in candidates if hint in str(p)]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        if supplied is None:
            archives = [p for root in INPUT_ROOTS if root.exists() for p in root.rglob('*.zip')]
        archive_matches = []
        for archive in sorted(set(archives)):
            try:
                with zipfile.ZipFile(archive) as z:
                    for info in z.infolist():
                        if Path(info.filename).name == filename and not info.is_dir():
                            if hint is None or hint in str(archive) + '/' + info.filename:
                                archive_matches.append((archive, info.filename))
            except zipfile.BadZipFile:
                continue
        if len(archive_matches) > 1:
            raise ValueError('Several archive candidates; set an exact input path: ' + str(archive_matches))
        if archive_matches:
            archive, member = archive_matches[0]
            dest = ROOT / 'extracted_inputs' / hashlib.sha256(str(archive.resolve()).encode()).hexdigest()[:12]
            dest.mkdir(parents=True, exist_ok=True)
            metadata_path = dest / 'extraction_record.json'
            records = read_json(metadata_path) if metadata_path.exists() else {}
            with zipfile.ZipFile(archive) as z:
                wanted = [member]
                sibling = str(Path(member).parent / 'cache_manifest.json') if filename == 'tokenizer_state.pkl' else str(Path(member).parent / 'sources/architecture_models.py')
                if sibling in z.namelist():
                    wanted.append(sibling)
                for name in wanted:
                    info = z.getinfo(name)
                    target = (dest / name).resolve()
                    if not target.is_relative_to(dest.resolve()):
                        raise ValueError('Unsafe path inside attached ZIP.')
                    record = {'archive_bytes': archive.stat().st_size, 'archive_mtime_ns': archive.stat().st_mtime_ns,
                              'member_crc': info.CRC, 'member_bytes': info.file_size}
                    if target.exists() and target.stat().st_size == info.file_size and records.get(name) == record:
                        continue
                    if shutil.disk_usage(ROOT).free < info.file_size + 256 * 2**20:
                        raise RuntimeError('Insufficient working disk to extract this selected Phase 1 file.')
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix(target.suffix + '.tmp')
                    with z.open(name) as src, open(temporary, 'wb') as dst:
                        shutil.copyfileobj(src, dst, length=2**20)
                    temporary.replace(target)
                    records[name] = record
                    print('Extracted selected input:', archive.name, '->', name)
            write_json(metadata_path, records)
            return dest / member
    if not candidates and not required:
        return None
    if not candidates:
        raise FileNotFoundError(f'Cannot find {filename}. Attach its extracted input folder or set an exact path.')
    raise ValueError(f'Several candidates for {filename}; set an exact path:\n' + '\n'.join(map(str, candidates)))

def official_labels(obj):
    if isinstance(obj, list):
        result = obj
    elif isinstance(obj, dict) and obj:
        if all(isinstance(v, int) and not isinstance(v, bool) for v in obj.values()):
            if set(obj.values()) != set(range(13)):
                raise ValueError('Expected label IDs 0..12 in label_vocab.json')
            result = [label for label, idx in sorted(obj.items(), key=lambda pair: pair[1])]
        elif all(isinstance(v, str) for v in obj.values()) and all(str(k).isdigit() for k in obj):
            if {int(k) for k in obj} != set(range(13)):
                raise ValueError('Expected label IDs 0..12 in label_vocab.json')
            result = [obj[str(i)] if str(i) in obj else obj[i] for i in range(13)]
        else:
            raise ValueError('Unsupported label_vocab.json mapping; inspect its preview below.')
    else:
        raise ValueError('Unsupported label vocabulary structure.')
    if len(result) != 13 or len(set(result)) != 13 or any(not isinstance(v, str) or not v.strip() for v in result):
        raise ValueError('Expected 13 unique nonempty label strings.')
    return result

def normalize_id(raw_id):
    if not isinstance(raw_id, str) or not raw_id.strip():
        raise ValueError('Document IDs must be nonempty strings.')
    return raw_id[:-4] if raw_id.endswith('.txt') else raw_id

def parse_documents(raw, domain, split, filename):
    if isinstance(raw, list):
        items = [(None, record) for record in raw]
    elif isinstance(raw, dict) and 'text' in raw and 'id' in raw:
        items = [(None, raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(f'{filename}: unsupported top-level structure.')
    result, seen = [], set()
    for key, record in items:
        if split == 'test' and key is not None and isinstance(record, list):
            record = {'id': key, 'text': record}
        if not isinstance(record, dict):
            raise ValueError(f'{filename}: expected records with id/text/labels; inspect the JSON preview.')
        raw_id = record.get('id', key)
        doc_id = normalize_id(raw_id)
        if key is not None and normalize_id(key) != doc_id:
            raise ValueError(f'{filename}: mapping key and record id disagree.')
        if doc_id in seen:
            raise ValueError(f'{filename}: duplicate normalized document ID {doc_id!r}')
        seen.add(doc_id)
        text = record.get('text')
        if not isinstance(text, list) or any(not isinstance(s, str) for s in text):
            raise ValueError(f'{filename}/{doc_id}: text must be an ordered list of strings.')
        labels = record.get('labels') if split != 'test' else None
        if split != 'test':
            if not isinstance(labels, list) or len(labels) != len(text):
                raise ValueError(f'{filename}/{doc_id}: sentence/label count mismatch.')
            if any(not isinstance(label, str) or label not in LABEL_TO_ID for label in labels):
                raise ValueError(f'{filename}/{doc_id}: unknown or non-string role label.')
        result.append({'domain': domain, 'split': split, 'file': filename, 'raw_id': raw_id,
                       'id': doc_id, 'text': text, 'labels': labels,
                       'test_labels_field_present': split == 'test' and 'labels' in record})
    return result

def sentence_windows(n, width, overlap):
    if not 0 <= overlap < width:
        raise ValueError('Require 0 <= overlap < width.')
    if n == 0:
        return []
    starts = [0]
    while starts[-1] + width < n:
        starts.append(starts[-1] + width - overlap)
    ends = [min(s + width, n) for s in starts]
    boundaries = [0] + [(ends[i-1] + starts[i]) // 2 for i in range(1, len(starts))] + [n]
    windows = [{'start': s, 'end': e, 'owned_start': boundaries[i], 'owned_end': boundaries[i+1]}
               for i, (s, e) in enumerate(zip(starts, ends))]
    ownership = np.zeros(n, dtype=np.int32)
    for w in windows:
        assert w['start'] <= w['owned_start'] <= w['owned_end'] <= w['end']
        ownership[w['owned_start']:w['owned_end']] += 1
    assert (ownership == 1).all()
    return windows

def split_chunks(ids, cap):
    if cap < 3:
        raise ValueError('Chunk cap must leave space for content plus two specials.')
    parts = [ids[i:i + cap - 2] for i in range(0, len(ids), cap - 2)] or [[]]
    return [[BOS_ID] + part + [EOS_ID] for part in parts]

PAD_ID, BOS_ID, EOS_ID = 0, 2, 3


def prepare_rr_data(destination, expected_hashes, revision, supplied=None):
    """Use verified local RR bytes when available; otherwise download seven files."""
    destination = Path(destination)
    relatives = {name: ('Test/' if name.endswith('_test.json') else 'Train/') + name
                 for name in expected_hashes}
    candidates = []
    if supplied is not None:
        p = Path(supplied)
        if not p.is_dir():
            raise ValueError('RR_DATA_INPUT must be an extracted data folder.')
        candidates = [p] + [f.parent.parent for f in p.rglob('CL_train.json')]
    else:
        candidates = [destination, Path('/kaggle/working/cs781_phase3_audit/official_data/Phase_3_RR')]
        candidates += [p.parent.parent for root in INPUT_ROOTS if root.exists()
                       for p in root.rglob('CL_train.json')]
    for candidate in candidates:
        if all((candidate / rel).exists() for rel in relatives.values()):
            if all(file_hash(candidate / relatives[name]) == digest for name, digest in expected_hashes.items()):
                return candidate
            if supplied is not None:
                raise ValueError('Selected RR data differs from the audited official revision.')
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import disable_progress_bars
    from kaggle_secrets import UserSecretsClient
    disable_progress_bars()
    try:
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('Enable Internet and the Kaggle secret HF_TOKEN, or attach verified RR files.') from None
    try:
        for name, rel in relatives.items():
            path = hf_hub_download(repo_id='Exploration-Lab/CS-781-Capstone', repo_type='dataset',
                filename='Phase_3_RR/' + rel, revision=revision, token=token,
                local_dir=str(destination.parent))
            if file_hash(path) != expected_hashes[name]:
                raise ValueError('Downloaded RR file differs from the audited hash: ' + name)
            print('RR data ready:', name, flush=True)
    except Exception:
        raise RuntimeError('RR download failed. Check Internet and HF_TOKEN access to the official dataset.') from None
    finally:
        token = None
    return destination


def load_rr_documents(folder):
    global LABEL_TO_ID
    folder = Path(folder)
    labels = official_labels(read_json(folder / 'Train/label_vocab.json'))
    LABEL_TO_ID = {name: i for i, name in enumerate(labels)}
    documents = []
    for domain in ('CL', 'IT'):
        for split in ('train', 'dev', 'test'):
            name = domain + '_' + split + '.json'
            raw = read_json(folder / ('Test' if split == 'test' else 'Train') / name)
            documents.extend(parse_documents(raw, domain, split, name))
    for split in ('train', 'dev', 'test'):
        ids = [d['id'] for d in documents if d['split'] == split]
        if len(ids) != len(set(ids)):
            raise ValueError('Duplicate normalized IDs across domains in ' + split)
    for a, b in (('train', 'dev'), ('train', 'test'), ('dev', 'test')):
        if {d['id'] for d in documents if d['split'] == a} & {d['id'] for d in documents if d['split'] == b}:
            raise ValueError('Document IDs overlap across splits.')
    return labels, documents


def load_tokenizer(path):
    from a1_tokenizer import A1BPETokenizer
    path = Path(path)
    manifest = read_json(path.parent / 'cache_manifest.json')
    if file_hash(path) != manifest['tokenizer_state_sha256']:
        raise ValueError('Tokenizer bytes differ from the preparation manifest.')
    with open(path, 'rb') as f:
        saved = pickle.load(f)
    if saved['signature'] != manifest['tokenizer_signature']:
        raise ValueError('Tokenizer signatures differ.')
    tokenizer = A1BPETokenizer()
    tokenizer.__dict__.update(saved['state'])
    assert tokenizer.vocab[:4] == [b'<pad>', b'<unk>', b'<s>', b'</s>']
    return tokenizer, saved['signature'], manifest


def tokenize_documents(documents, tokenizer, fingerprint, cache_dir):
    cache = Path(cache_dir) / (fingerprint + '.pkl')
    if cache.exists():
        with open(cache, 'rb') as f:
            saved = pickle.load(f)
        if saved['fingerprint'] != fingerprint or len(saved['tokens']) != len(documents):
            raise ValueError('RR token cache identity differs.')
        for doc, tokens, identity in zip(documents, saved['tokens'], saved['identities']):
            if identity != [doc['file'], doc['raw_id']] or len(tokens) != len(doc['text']):
                raise ValueError('RR token cache sentence alignment differs.')
            doc['token_ids'] = tokens
        print('Loaded verified RR token cache.', flush=True)
        return
    from functools import lru_cache
    @lru_cache(maxsize=100000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    started = last = time.monotonic()
    for i, doc in enumerate(documents):
        doc['token_ids'] = [[idx for p in re.findall(r'\S+|\s+', sentence) for idx in piece_ids(p)]
                            for sentence in doc['text']]
        if i == 0 or time.monotonic() - last > 30:
            print('Tokenized RR documents:', i + 1, '/', len(documents), flush=True)
            last = time.monotonic()
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache.with_suffix('.tmp'), 'wb') as f:
        pickle.dump({'fingerprint': fingerprint, 'tokens': [d['token_ids'] for d in documents],
                     'identities': [[d['file'], d['raw_id']] for d in documents]}, f, protocol=pickle.HIGHEST_PROTOCOL)
    cache.with_suffix('.tmp').replace(cache)
    print('Tokenization minutes:', round((time.monotonic() - started) / 60, 2), flush=True)


def make_windows(documents, width, overlap, split):
    result = []
    for di, doc in enumerate(documents):
        if doc['split'] != split:
            continue
        for w in sentence_windows(len(doc['text']), width, overlap):
            result.append(dict(w, doc_index=di, n_document_sentences=len(doc['text'])))
    return result


def class_weights(documents, labels, cap=5.0):
    counts = Counter(label for d in documents if d['split'] == 'train' for label in d['labels'])
    maximum = max(counts.values())
    weights = [min(math.sqrt(maximum / counts[k]), cap) if counts[k] else 0.0 for k in labels]
    return torch.tensor(weights, dtype=torch.float32), [counts[k] for k in labels]


def collate_windows(documents, windows, label_to_id, cap):
    rows = []
    for w in windows:
        doc = documents[w['doc_index']]
        chunks, owners, weights = [], [], []
        for owner, ids in enumerate(doc['token_ids'][w['start']:w['end']]):
            for chunk in split_chunks(ids, cap):
                chunks.append(chunk)
                owners.append(owner)
                weights.append(max(len(chunk) - 2, 1))
        rows.append((chunks, owners, weights, doc, w))
    if not rows:
        raise ValueError('Cannot collate an empty batch.')
    batch_size = len(rows)
    max_chunks = max(len(r[0]) for r in rows)
    max_tokens = max(len(c) for r in rows for c in r[0])
    max_sentences = max(r[4]['end'] - r[4]['start'] for r in rows)
    batch = {
        'ids': torch.full((batch_size, max_chunks, max_tokens), PAD_ID, dtype=torch.long),
        'lengths': torch.zeros(batch_size, max_chunks, dtype=torch.long),
        'owners': torch.zeros(batch_size, max_chunks, dtype=torch.long),
        'chunk_weights': torch.zeros(batch_size, max_chunks, dtype=torch.float32),
        'sentence_mask': torch.zeros(batch_size, max_sentences, dtype=torch.bool),
        'owned_mask': torch.zeros(batch_size, max_sentences, dtype=torch.bool),
        'targets': torch.full((batch_size, max_sentences), -100, dtype=torch.long),
        'use_start': torch.zeros(batch_size, dtype=torch.bool),
        'use_end': torch.zeros(batch_size, dtype=torch.bool),
    }
    for b, (chunks, owners, weights, doc, w) in enumerate(rows):
        n = w['end'] - w['start']
        batch['sentence_mask'][b, :n] = True
        lo, hi = w['owned_start'] - w['start'], w['owned_end'] - w['start']
        batch['owned_mask'][b, lo:hi] = True
        batch['use_start'][b] = w['owned_start'] == 0
        batch['use_end'][b] = w['owned_end'] == len(doc['text'])
        if doc['labels'] is not None:
            batch['targets'][b, :n] = torch.tensor([label_to_id[k] for k in doc['labels'][w['start']:w['end']]])
        for j, chunk in enumerate(chunks):
            batch['ids'][b, j, :len(chunk)] = torch.tensor(chunk)
            batch['lengths'][b, j] = len(chunk)
            batch['owners'][b, j] = owners[j]
            batch['chunk_weights'][b, j] = weights[j]
    return batch


def supervision_denominators(batch, weights):
    targets = batch['targets'][batch['owned_mask']]
    if (targets < 0).any():
        raise ValueError('An owned training position lacks a label.')
    return int(len(targets)), float(weights.cpu()[targets].sum())


def load_own_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    except RuntimeError as error:
        if 'mmap' not in str(error):
            raise
        return torch.load(path, map_location='cpu', weights_only=False)


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/rr_data.py


In [7]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/rr_shared.py
"""Established RR gates, unconstrained CRF, and trainable-state helpers."""
import torch
from torch import nn
import torch.nn.functional as F


class SentenceBiGRU(nn.Module):
    """One manually gated GRU per direction; reset-after convention."""
    def __init__(self, input_size, hidden):
        super().__init__()
        self.hidden = hidden
        self.x_layers = nn.ModuleList([nn.Linear(input_size, 3 * hidden) for _ in range(2)])
        self.h_layers = nn.ModuleList([nn.Linear(hidden, 3 * hidden) for _ in range(2)])

    def forward(self, x):
        outputs = []
        for direction in range(2):
            hidden = x.new_zeros(self.hidden)
            states = [None] * len(x)
            order = range(len(x)) if direction == 0 else range(len(x) - 1, -1, -1)
            for i in order:
                xr, xz, xn = self.x_layers[direction](x[i]).chunk(3)
                hr, hz, hn = self.h_layers[direction](hidden).chunk(3)
                reset, update = torch.sigmoid(xr + hr), torch.sigmoid(xz + hz)
                candidate = torch.tanh(xn + reset * hn)
                hidden = (1 - update) * candidate + update * hidden
                states[i] = hidden
            outputs.append(torch.stack(states))
        return torch.cat(outputs, dim=-1)


class LinearChainCRF(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(n_labels, n_labels))
        self.start = nn.Parameter(torch.zeros(n_labels))
        self.end = nn.Parameter(torch.zeros(n_labels))

    def log_partition(self, emissions, use_start=True, use_end=True):
        if len(emissions) == 0:
            return emissions.sum() * 0
        alpha = emissions[0] + (self.start if use_start else 0)
        for emission in emissions[1:]:
            alpha = emission + torch.logsumexp(alpha[:, None] + self.transitions, dim=0)
        return torch.logsumexp(alpha + (self.end if use_end else 0), dim=0)

    def gold_score(self, emissions, labels, use_start=True, use_end=True):
        if len(labels) == 0:
            return emissions.sum() * 0
        score = emissions.gather(1, labels[:, None]).sum()
        if len(labels) > 1:
            score = score + self.transitions[labels[:-1], labels[1:]].sum()
        if use_start:
            score = score + self.start[labels[0]]
        if use_end:
            score = score + self.end[labels[-1]]
        return score

    def nll(self, emissions, labels, use_start=True, use_end=True):
        return self.log_partition(emissions, use_start, use_end) - self.gold_score(emissions, labels, use_start, use_end)

    @torch.no_grad()
    def decode(self, emissions, use_start=True, use_end=True):
        if len(emissions) == 0:
            return []
        delta = emissions[0] + (self.start if use_start else 0)
        back = []
        for emission in emissions[1:]:
            scores, choices = (delta[:, None] + self.transitions).max(dim=0)
            delta = scores + emission
            back.append(choices)
        last = int((delta + (self.end if use_end else 0)).argmax())
        path = [last]
        for choices in reversed(back):
            last = int(choices[last])
            path.append(last)
        return list(reversed(path))


def bare(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in bare(model).named_parameters() if p.requires_grad}


def load_trainable_state(model, state):
    expected = {name: p for name, p in bare(model).named_parameters() if p.requires_grad}
    if set(expected) != set(state):
        raise ValueError('Checkpoint trainable parameter names differ.')
    with torch.no_grad():
        for name, p in expected.items():
            if tuple(p.shape) != tuple(state[name].shape):
                raise ValueError('Checkpoint parameter shape differs: ' + name)
            p.copy_(state[name].to(device=p.device, dtype=p.dtype))


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/rr_shared.py


In [8]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/rr_check_utils.py
import itertools
import torch
import torch.nn.functional as F
from rr_shared import LinearChainCRF


def crf_self_test():
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(19)
        crf = LinearChainCRF(3).double()
        with torch.no_grad():
            for p in crf.parameters():
                p.normal_(0, 0.2)
        emissions = torch.randn(4, 3, dtype=torch.float64, requires_grad=True)
        for use_start, use_end in ((True, True), (False, False), (True, False), (False, True)):
            paths = [torch.tensor(p) for p in itertools.product(range(3), repeat=4)]
            scores = torch.stack([crf.gold_score(emissions, p, use_start, use_end) for p in paths])
            torch.testing.assert_close(crf.log_partition(emissions, use_start, use_end), torch.logsumexp(scores, 0))
            expected = paths[int(scores.argmax())].tolist()
            assert crf.decode(emissions, use_start, use_end) == expected
            crf.nll(emissions, paths[0], use_start, use_end).backward(retain_graph=True)
            assert torch.isfinite(emissions.grad).all()
        zero = LinearChainCRF(3)
        one = torch.tensor([[0.2, -0.1, 0.8]])
        torch.testing.assert_close(zero.nll(one, torch.tensor([2])), F.cross_entropy(one, torch.tensor([2])))
        assert zero.decode(one) == [2]
    return {'status': 'passed', 'checks': ['partition vs exhaustive enumeration', 'Viterbi vs exhaustive search',
                                          'all four start/end boundary combinations', 'finite gradients', 'length-one CE identity']}


def learning_windows(documents, width=8):
    candidates = []
    for di, doc in enumerate(documents):
        if doc['split'] != 'train' or len(doc['text']) < width:
            continue
        for start in range(0, len(doc['text']) - width + 1, max(width // 2, 1)):
            labels = doc['labels'][start:start + width]
            changes = sum(a != b for a, b in zip(labels, labels[1:]))
            score = (len(set(labels)), changes)
            candidates.append((score, {'doc_index': di, 'start': start, 'end': start + width,
                'owned_start': start, 'owned_end': start + width, 'n_document_sentences': len(doc['text'])}))
    candidates.sort(key=lambda pair: pair[0], reverse=True)
    selected = []
    for _, w in candidates:
        if not selected or w['doc_index'] != selected[0]['doc_index']:
            selected.append(w)
        if len(selected) == 2:
            break
    if not selected:
        raise ValueError('No training windows available for the learning check.')
    if len(selected) == 1:
        selected.append(dict(selected[0]))
    return selected


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/rr_check_utils.py


In [9]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/rr_encdec_model.py
"""Encoder–decoder LoRA and sentence-level RR prototype for preflight only."""
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from architecture_models import valid_keys
from rr_shared import SentenceBiGRU, LinearChainCRF


class LoRALinear(nn.Module):
    def __init__(self, original, rank, alpha, dropout):
        super().__init__()
        if not isinstance(original, nn.Linear) or not 0 < rank <= min(original.weight.shape):
            raise ValueError('Invalid LoRA target or rank.')
        self.original = original
        for p in original.parameters():
            p.requires_grad_(False)
        self.lora_A = nn.Parameter(original.weight.new_empty(rank, original.in_features))
        self.lora_B = nn.Parameter(original.weight.new_zeros(original.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = float(alpha) / rank
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        delta = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B)
        return self.original(x) + self.scaling * delta


def token_features(lm, ids, lengths, use_checkpoint=False):
    """Both stacks consume known text. Original encoder/decoder masks are retained."""
    if 2 * ids.shape[1] > lm.cfg.max_seq_len:
        raise ValueError('Encoder plus decoder input exceeds the original context budget.')
    allowed = valid_keys(lengths, ids.shape[1])[:, None, None, :]
    memory = lm.embed(ids)
    for block in lm.encoder:
        memory = checkpoint(block, memory, allowed, use_reentrant=False) if use_checkpoint else block(memory, allowed)
    memory = lm.encoder_norm(memory)
    x = lm.embed(ids.masked_fill(~valid_keys(lengths, ids.shape[1]), 0))
    for block in lm.decoder:
        x = checkpoint(block, x, memory, allowed, use_reentrant=False) if use_checkpoint else block(x, memory, allowed)
    return lm.decoder_norm(x)


def content_mean(states, lengths):
    positions = torch.arange(states.shape[1], device=states.device)[None, :]
    mask = (positions > 0) & (positions < lengths[:, None] - 1)
    count = mask.sum(1)
    pooled = (states.float() * mask[:, :, None]).sum(1) / count.clamp_min(1)[:, None]
    return torch.where((count > 0)[:, None], pooled, states[:, 0].float())


class EncoderDecoderRR(nn.Module):
    def __init__(self, lm, n_labels, weights, settings):
        super().__init__()
        if lm.cfg.architecture != 'encoder_decoder':
            raise ValueError('Use the original Phase 1 encoder–decoder checkpoint.')
        self.lm, self.settings, self.n_labels = lm, dict(settings), n_labels
        for p in lm.parameters():
            p.requires_grad_(False)
        self.adapter_targets = []
        for i, block in enumerate(lm.encoder):
            self._attention(block.attn, f'lm.encoder.{i}.attn')
            self._ffn(block.ffn, f'lm.encoder.{i}.ffn')
        for i, block in enumerate(lm.decoder):
            for name in ('self_attn', 'cross_attn'):
                self._attention(getattr(block, name), f'lm.decoder.{i}.{name}')
            self._ffn(block.ffn, f'lm.decoder.{i}.ffn')
        self.projection = nn.Linear(lm.cfg.d_model, settings['projection_dim'])
        self.context = SentenceBiGRU(settings['projection_dim'], settings['context_hidden'])
        self.dropout = nn.Dropout(settings['head_dropout'])
        self.classifier = nn.Linear(2 * settings['context_hidden'], n_labels)
        self.crf = LinearChainCRF(n_labels)
        self.register_buffer('class_weights', weights.clone().float(), persistent=False)
        for name, p in self.named_parameters():
            expected = name.endswith(('.lora_A', '.lora_B')) or not name.startswith('lm.')
            if p.requires_grad != expected:
                raise AssertionError('Unexpected trainability: ' + name)

    def _replace(self, original, path):
        self.adapter_targets.append(path)
        return LoRALinear(original, self.settings['lora_rank'], self.settings['lora_alpha'], self.settings['lora_dropout'])

    def _attention(self, attention, path):
        for name in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            setattr(attention, name, self._replace(getattr(attention, name), path + '.' + name))

    def _ffn(self, ffn, path):
        for index in (0, 2):
            ffn.net[index] = self._replace(ffn.net[index], f'{path}.net.{index}')

    def encode_chunks(self, ids, lengths):
        return content_mean(token_features(self.lm, ids, lengths,
            self.training and self.settings['gradient_checkpointing']), lengths)

    def one_window(self, ids, lengths, owners, weights, n_sentences):
        sizes = lengths.detach().cpu().tolist()
        n_chunks = sum(n > 0 for n in sizes)
        if n_chunks == 0:
            raise ValueError('Empty RR window.')
        vectors = []
        micro = self.settings['chunk_microbatch']
        for start in range(0, n_chunks, micro):
            end = min(start + micro, n_chunks)
            vectors.append(self.encode_chunks(ids[start:end, :max(sizes[start:end])], lengths[start:end]))
        vectors = torch.cat(vectors).float()
        weights = weights[:n_chunks].float()
        pooled = vectors.new_zeros((n_sentences, vectors.shape[-1])).index_add(
            0, owners[:n_chunks], vectors * weights[:, None])
        denominator = vectors.new_zeros(n_sentences).index_add(0, owners[:n_chunks], weights)
        features = self.dropout(torch.tanh(self.projection(pooled / denominator[:, None])))
        return self.classifier(self.dropout(self.context(features)))

    def forward(self, ids, lengths, owners, chunk_weights, sentence_mask, owned_mask,
                targets, use_start, use_end, mode='loss'):
        results = []
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=ids.is_cuda):
            for b in range(len(ids)):
                n = int(sentence_mask[b].sum())
                logits = self.one_window(ids[b], lengths[b], owners[b], chunk_weights[b], n)
                if mode == 'emissions':
                    results.append(F.pad(logits.float(), (0, 0, 0, sentence_mask.shape[1] - n)))
                    continue
                gold = targets[b, :n][owned_mask[b, :n]]
                scores = logits[owned_mask[b, :n]].float()
                if len(gold) == 0 or (gold < 0).any():
                    raise ValueError('Owned training sentences require gold labels.')
                with torch.autocast(device_type='cuda', enabled=False):
                    crf = self.crf.nll(scores, gold, bool(use_start[b]), bool(use_end[b]))
                    ce = F.cross_entropy(scores, gold, weight=self.class_weights, reduction='sum')
                device_id = ids.device.index if ids.is_cuda else -1
                results.append(torch.stack((crf, ce, crf.new_tensor(float(device_id)))))
        return torch.stack(results)


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/rr_encdec_model.py


In [10]:
%%writefile /kaggle/working/cs781_phase3_encdec_preflight_src/rr_encdec_preflight.py
"""Bounded correctness, temporary learning and GPU resource checks; no training run."""
import copy
import gc
import math
from pathlib import Path
import random
import statistics
import time
import traceback
import zipfile

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from rr_data import collate_windows, supervision_denominators, make_windows, split_chunks, write_json
from rr_shared import bare, trainable_state, load_trainable_state
from rr_check_utils import crf_self_test, learning_windows


def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    for i, saved in enumerate(state['cuda'][:torch.cuda.device_count()]):
        torch.cuda.set_rng_state(saved.cpu(), i)


def synchronize():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)


def clean(model):
    model.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def feed(model, batch, mode='loss'):
    if isinstance(model, nn.DataParallel):
        return model(**batch, mode=mode)
    device = next(model.parameters()).device
    return model(**{k: v.to(device) for k, v in batch.items()}, mode=mode)


def loss_for(stats, count, weight_sum, settings):
    return stats[:, 0].sum() / count + settings['aux_ce_coefficient'] * stats[:, 1].sum() / weight_sum


def optimizer_for(model, settings, multiplier=1.0):
    groups = {}
    for name, p in bare(model).named_parameters():
        if not p.requires_grad:
            continue
        if name.endswith(('.lora_A', '.lora_B')):
            kind, lr, decay = 'lora', settings['lora_lr'], 0.0
        elif name.startswith('crf.'):
            kind, lr, decay = 'crf', settings['crf_lr'], 0.0
        else:
            kind, lr = 'head', settings['head_lr']
            decay = settings['weight_decay'] if p.ndim > 1 else 0.0
        groups.setdefault((kind, decay), {'params': [], 'lr': lr * multiplier,
                                         'weight_decay': decay, 'kind': kind})['params'].append(p)
    return torch.optim.AdamW(list(groups.values()), betas=(0.9, 0.999), eps=1e-8, foreach=False)


def gradients(model):
    totals = {name: {'A': 0.0, 'B': 0.0} for name in
              ('encoder_attention', 'encoder_ffn', 'decoder_self', 'decoder_cross', 'decoder_ffn')}
    finite, frozen = True, 0
    for name, p in bare(model).named_parameters():
        if not p.requires_grad:
            frozen += int(p.grad is not None)
            continue
        if p.grad is not None:
            finite = finite and bool(torch.isfinite(p.grad).all())
        if not name.endswith(('.lora_A', '.lora_B')):
            continue
        if p.grad is None:
            raise AssertionError('Missing LoRA gradient path: ' + name)
        if '.encoder.' in name:
            group = 'encoder_ffn' if '.ffn.' in name else 'encoder_attention'
        elif '.ffn.' in name:
            group = 'decoder_ffn'
        else:
            group = 'decoder_cross' if '.cross_attn.' in name else 'decoder_self'
        totals[group][name[-1]] += float(p.grad.detach().float().abs().sum())
    if frozen or not finite:
        raise AssertionError('Frozen parameters received gradients or trainable gradients are nonfinite.')
    return totals


@torch.no_grad()
def mini_metrics(model, batch):
    model.eval()
    emissions = feed(model, batch, mode='emissions').float().cpu()
    keep = batch['owned_mask']
    scores, target = emissions[keep], batch['targets'][keep]
    return {'unweighted_ce': float(F.cross_entropy(scores, target)),
            'emission_accuracy': float((scores.argmax(1) == target).float().mean())}


def run_core_checks(model, documents, labels, weights, settings, output_dir, learning_steps=40):
    output_dir = Path(output_dir)
    initial, initial_rng = trainable_state(model), rng_state()
    initial_mode = model.training
    report = {'crf': crf_self_test()}
    optimizer = None
    try:
        if settings['require_two_gpus'] and (not isinstance(model, nn.DataParallel) or len(model.device_ids) != 2):
            raise AssertionError('Select T4 x2 for this preflight.')
        mini = learning_windows(documents)
        batch = collate_windows(documents, mini, {v: i for i, v in enumerate(labels)}, settings['chunk_tokens'])
        count, weight_sum = supervision_denominators(batch, weights)
        model.eval()
        model.zero_grad(set_to_none=True)
        stats = feed(model, batch)
        loss = loss_for(stats, count, weight_sum, settings)
        (128 * loss).backward()
        first_grads = gradients(model)
        if any(v['B'] <= 0 or v['A'] != 0 for v in first_grads.values()):
            raise AssertionError('At zero B initialization, expect active B paths and zero A gradients in every stack.')
        seen = sorted(set(int(i) for i in stats[:, 2].detach().cpu().tolist()))
        report['devices_seen'] = seen
        if isinstance(model, nn.DataParallel):
            if seen != sorted(model.device_ids):
                raise AssertionError('Both GPUs did not participate.')
            parallel = {name: (p.grad if p.grad is not None else torch.zeros_like(p)).detach().float().cpu().clone() / 128
                        for name, p in bare(model).named_parameters() if p.requires_grad}
            model.zero_grad(set_to_none=True)
            serial = feed(bare(model), batch)
            serial_loss = loss_for(serial, count, weight_sum, settings)
            (128 * serial_loss).backward()
            numerator, denominator = 0.0, 0.0
            for name, p in bare(model).named_parameters():
                if name in parallel:
                    ref = (p.grad if p.grad is not None else torch.zeros_like(p)).detach().float().cpu() / 128
                    numerator += float((ref - parallel[name]).square().sum())
                    denominator += float(ref.square().sum())
            relative = math.sqrt(numerator / max(denominator, 1e-20))
            difference = abs(float(loss.detach()) - float(serial_loss.detach()))
            report['parallel_vs_serial'] = {'loss_difference': difference, 'gradient_relative_l2_error': relative}
            if relative > 0.03 or difference > 0.01:
                raise AssertionError('Parallel and serial gradient normalization differ.')
            del parallel, serial, serial_loss
        del stats, loss
        clean(model)
        load_trainable_state(model, initial)

        # Two small updates distinguish zero-at-initialization from a broken path.
        optimizer = optimizer_for(model, settings)
        for step in range(2):
            model.eval()
            optimizer.zero_grad(set_to_none=True)
            stats = feed(model, batch)
            loss = loss_for(stats, count, weight_sum, settings)
            (128 * loss).backward()
            for p in bare(model).parameters():
                if p.grad is not None:
                    p.grad.div_(128)
            paths = gradients(model)
            if step == 1 and any(v['A'] <= 0 or v['B'] <= 0 for v in paths.values()):
                raise AssertionError('An encoder/decoder LoRA component has no active A/B gradients after an update.')
            nn.utils.clip_grad_norm_([p for p in bare(model).parameters() if p.requires_grad], settings['gradient_clip'])
            optimizer.step()
        report['gradient_paths'] = {'status': 'passed', 'first_backward': first_grads, 'second_backward': paths}
        del optimizer, stats, loss
        optimizer = None
        clean(model)
        load_trainable_state(model, initial)

        before = mini_metrics(model, batch)
        optimizer = optimizer_for(model, settings, multiplier=5.0)
        scaler = torch.amp.GradScaler('cuda', enabled=next(model.parameters()).is_cuda, init_scale=1024)
        skipped = 0
        for step in range(learning_steps):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            stats = feed(model, batch)
            loss = loss_for(stats, count, weight_sum, settings)
            if not bool(torch.isfinite(loss)):
                raise AssertionError('Nonfinite temporary learning loss.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            gradients(model)
            nn.utils.clip_grad_norm_([p for p in bare(model).parameters() if p.requires_grad], settings['gradient_clip'])
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old_scale)
            if step == 0 or (step + 1) % 10 == 0:
                print('Temporary learning:', step + 1, '/', learning_steps, '| loss', round(float(loss.detach()), 5), flush=True)
        after = mini_metrics(model, batch)
        passed = after['unweighted_ce'] < 0.85 * before['unweighted_ce'] and (
            after['emission_accuracy'] >= before['emission_accuracy'] + 0.10 or after['emission_accuracy'] >= 0.90)
        report['learning'] = {'before': before, 'after': after, 'steps': learning_steps,
                              'skipped_updates': skipped, 'windows': mini, 'rate_multiplier': 5,
                              'status': 'passed' if passed else 'failed'}
        if not passed:
            raise AssertionError('Temporary learning did not improve enough; inspect this diagnostic before training.')
        report['status'] = 'passed'
    except Exception as error:
        report.update(status='failed', failure=str(error), traceback=traceback.format_exc())
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        clean(model)
        load_trainable_state(model, initial)
        restore_rng(initial_rng)
        model.train(initial_mode)
        report['initial_state_restored'] = all(torch.equal(v, initial[k]) for k, v in trainable_state(model).items())
        if not report['initial_state_restored']:
            report.update(status='failed', failure='Temporary updates were not fully restored.')
        write_json(output_dir / 'core_checks.json', report)
    return report


def window_cost(documents, window, cap):
    ids = documents[window['doc_index']]['token_ids'][window['start']:window['end']]
    return (sum(max(1, math.ceil(len(v) / (cap - 2))) for v in ids), sum(map(len, ids)))


def describe_window(documents, w, cap):
    return dict(w, document_id=documents[w['doc_index']]['raw_id'],
                chunks=window_cost(documents, w, cap)[0], content_tokens=window_cost(documents, w, cap)[1])


def update_group(model, batches, weights, settings, optimizer, scaler):
    denominators = [supervision_denominators(b, weights) for b in batches]
    count, weight_sum = sum(v[0] for v in denominators), sum(v[1] for v in denominators)
    optimizer.zero_grad(set_to_none=True)
    synchronize()
    started = time.perf_counter()
    for batch in batches:
        stats = feed(model, batch)
        loss = loss_for(stats, count, weight_sum, settings)
        if not bool(torch.isfinite(loss)):
            raise AssertionError('Nonfinite probe loss.')
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    norm = nn.utils.clip_grad_norm_([p for p in bare(model).parameters() if p.requires_grad], settings['gradient_clip'])
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    skipped = scaler.get_scale() < old_scale
    if not bool(torch.isfinite(norm)) and not skipped:
        raise AssertionError('Nonfinite probe gradient norm without an AMP skip.')
    synchronize()
    return {'seconds': time.perf_counter() - started, 'amp_skipped': skipped}


def probe_case(model, documents, labels, weights, settings, width, overlap, micro, timed_updates=2):
    initial, saved_rng = trainable_state(model), rng_state()
    original_settings, original_mode = dict(bare(model).settings), model.training
    current = dict(settings, window_sentences=width, overlap_sentences=overlap, chunk_microbatch=micro)
    bare(model).settings.update(current)
    row = {'window_sentences': width, 'overlap_sentences': overlap, 'chunk_microbatch': micro,
           'gradient_checkpointing': current['gradient_checkpointing'], 'accumulation_steps': current['accumulation_steps']}
    optimizer = None
    try:
        windows = make_windows(documents, width, overlap, 'train')
        group_size = current['global_window_batch'] * current['accumulation_steps']
        heavy = sorted(windows, key=lambda w: window_cost(documents, w, current['chunk_tokens']), reverse=True)[:group_size]
        heavy = [heavy[i % len(heavy)] for i in range(group_size)]
        heavy_batches = [collate_windows(documents, heavy[i:i+current['global_window_batch']],
                         {v:j for j,v in enumerate(labels)}, current['chunk_tokens'])
                         for i in range(0, group_size, current['global_window_batch'])]
        row['heavy_windows'] = [describe_window(documents, w, current['chunk_tokens']) for w in heavy]
        clean(model)
        optimizer = optimizer_for(model, current)
        scaler = torch.amp.GradScaler('cuda', enabled=next(model.parameters()).is_cuda, init_scale=1024)
        for device in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(device)
        model.train()
        skipped = 0
        for attempt in range(4):
            warm = update_group(model, heavy_batches, weights, current, optimizer, scaler)
            skipped += int(warm['amp_skipped'])
            if not warm['amp_skipped']:
                break
        else:
            raise AssertionError('No completed warmup optimizer update after AMP retries.')
        timings = []
        for _ in range(timed_updates):
            measured = update_group(model, heavy_batches, weights, current, optimizer, scaler)
            skipped += int(measured['amp_skipped'])
            if measured['amp_skipped']:
                raise AssertionError('A timed update was skipped; this timing is not reliable.')
            timings.append(measured['seconds'])
        row.update(status='passed', warmup_seconds=warm['seconds'], heavy_update_seconds=timings,
            median_heavy_update_seconds=statistics.median(timings), amp_skipped_updates=skipped,
            optimizer_state_parameters=len(optimizer.state),
            peak_allocated_gib=[torch.cuda.max_memory_allocated(i)/2**30 for i in range(torch.cuda.device_count())],
            peak_reserved_gib=[torch.cuda.max_memory_reserved(i)/2**30 for i in range(torch.cuda.device_count())])
    except Exception as error:
        oom = isinstance(error, torch.cuda.OutOfMemoryError) or 'out of memory' in str(error).lower()
        row.update(status='oom' if oom else 'failed', failure=str(error), traceback=traceback.format_exc())
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        optimizer = None
        clean(model)
        load_trainable_state(model, initial)
        bare(model).settings.clear()
        bare(model).settings.update(original_settings)
        restore_rng(saved_rng)
        model.train(original_mode)
    return row


@torch.no_grad()
def time_dev_forward(model, documents, labels, settings):
    model.eval()
    windows = make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'dev')
    synchronize()
    started = time.perf_counter()
    for start in range(0, len(windows), settings['global_window_batch']):
        batch = collate_windows(documents, windows[start:start+settings['global_window_batch']],
                                {v:i for i,v in enumerate(labels)}, settings['chunk_tokens'])
        _ = feed(model, batch, mode='emissions')
    synchronize()
    return time.perf_counter() - started


def representative_timing(model, documents, labels, weights, settings, selected, timed_updates=3):
    initial, saved_rng = trainable_state(model), rng_state()
    original_settings, original_mode = dict(bare(model).settings), model.training
    current = dict(settings, **{k:selected[k] for k in ('window_sentences','overlap_sentences','chunk_microbatch')})
    bare(model).settings.update(current)
    optimizer = None
    try:
        windows = make_windows(documents, current['window_sentences'], current['overlap_sentences'], 'train')
        order = np.random.default_rng(current['seed']).permutation(len(windows))
        group_size = current['global_window_batch'] * current['accumulation_steps']
        optimizer = optimizer_for(model, current)
        scaler = torch.amp.GradScaler('cuda', enabled=next(model.parameters()).is_cuda, init_scale=1024)
        times = []
        for group in range(timed_updates + 1):
            chosen = [windows[int(order[(group*group_size+i) % len(order)])] for i in range(group_size)]
            batches = [collate_windows(documents, chosen[i:i+current['global_window_batch']],
                        {v:j for j,v in enumerate(labels)}, current['chunk_tokens'])
                        for i in range(0,group_size,current['global_window_batch'])]
            model.train()
            value = update_group(model,batches,weights,current,optimizer,scaler)
            if value['amp_skipped']:
                raise AssertionError('Representative update was skipped; timing requires review.')
            if group:
                times.append(value['seconds'])
        optimizer.zero_grad(set_to_none=True)
        optimizer = None
        clean(model)
        dev_seconds = time_dev_forward(model,documents,labels,current)
        dev_windows = make_windows(documents,current['window_sentences'],current['overlap_sentences'],'dev')
        train_work=sum(window_cost(documents,w,current['chunk_tokens'])[0] for w in windows)
        dev_work=sum(window_cost(documents,w,current['chunk_tokens'])[0] for w in dev_windows)
        updates=math.ceil(math.ceil(len(windows)/current['global_window_batch'])/current['accumulation_steps'])
        training_seconds=statistics.mean(times)*updates
        approximate_full_train_eval=dev_seconds*train_work/max(dev_work,1)
        epoch=training_seconds+dev_seconds+approximate_full_train_eval/3
        return {'status':'passed','representative_update_seconds':times,'mean_update_seconds':statistics.mean(times),
            'train_windows':len(windows),'updates_per_epoch':updates,'dev_forward_seconds':dev_seconds,
            'estimated_training_seconds_per_epoch':training_seconds,
            'estimated_full_train_eval_seconds':approximate_full_train_eval,
            'estimated_average_epoch_minutes':epoch/60,'estimated_35_epoch_hours':epoch*35/3600,
            'notes':'Rough estimate from three sampled updates, all dev forward passes, and chunk-scaled train inference every three epochs. Excludes CRF decoding, metrics, checkpoint I/O and setup; early stopping may shorten the run.'}
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        optimizer=None
        clean(model)
        load_trainable_state(model,initial)
        bare(model).settings.clear()
        bare(model).settings.update(original_settings)
        restore_rng(saved_rng)
        model.train(original_mode)


def run_resource_probes(model, documents, labels, weights, settings, output_dir):
    rows=[]
    def attempt(width,overlap,micro):
        print(f'Heavy probe: window {width}, overlap {overlap}, chunk microbatch {micro}',flush=True)
        row=probe_case(model,documents,labels,weights,settings,width,overlap,micro)
        rows.append(row)
        write_json(Path(output_dir)/'resource_probes.json',rows)
        print('Result:',row['status'],'| median update seconds:',row.get('median_heavy_update_seconds'),flush=True)
        if row['status']=='failed':
            raise RuntimeError('Resource probe failed for a reason other than memory. Inspect resource_probes.json.')
        return row
    base=attempt(128,32,8)
    if base['status']=='passed':
        attempt(128,32,16)
    else:
        smaller=attempt(128,32,4)
        if smaller['status']!='passed':
            attempt(64,16,4)
    passed=[r for r in rows if r['status']=='passed']
    if not passed:
        return {'status':'failed','failure':'None of the bounded resource configurations fit.','cases':rows}
    totals=[torch.cuda.get_device_properties(i).total_memory/2**30 for i in range(torch.cuda.device_count())]
    headroom=[r for r in passed if all(peak < 0.85*total for peak,total in zip(r['peak_reserved_gib'],totals))]
    candidates=headroom or passed
    widest=max(r['window_sentences'] for r in candidates)
    candidates=[r for r in candidates if r['window_sentences']==widest]
    selected=min(candidates,key=lambda r:r['median_heavy_update_seconds'])
    chosen={k:selected[k] for k in ('window_sentences','overlap_sentences','chunk_microbatch','gradient_checkpointing')}
    timing=representative_timing(model,documents,labels,weights,settings,selected)
    return {'status':'passed','cases':rows,'recommended_configuration':chosen,
            'memory_headroom_target_met':bool(headroom),'timing':timing}


def diagnostic_zip(run_dir):
    run_dir=Path(run_dir)
    destination=run_dir.parent/(run_dir.name+'_diagnostics.zip')
    with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(run_dir.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.csv','.py','.txt'):
                z.write(p,arcname=run_dir.name+'/'+str(p.relative_to(run_dir)))
    return destination


Overwriting /kaggle/working/cs781_phase3_encdec_preflight_src/rr_encdec_preflight.py


## 3. Verify the data and original checkpoint

The notebook verifies the seven `Phase_3_RR` files at the pinned revision and reuses the 16K byte BPE tokenizer from Phase 1. I trained on the official CL and IT training sets together and kept the official document-level dev and test splits. The scorer uses a fixed vocabulary of 13 roles. Two roles are absent from combined dev, giving a perfect-prediction dev ceiling of 11/13.


In [11]:
HF_REVISION='89edba6a44d6bb9593fd13b6895b5cd3b06eb82e'
EXPECTED_RR_HASHES={'CL_train.json': '2fdad020443ad33734a8c3a59f4fbae72582e0535692708ad902473087333ed1', 'CL_dev.json': '95ec2dac8b4c869f9fd1ae434546103b93d9c0a5340b2e5a7e611773519dbbcc', 'IT_train.json': 'a9111869c768c1adfb492ba03df72b14d77fcce9f468a2a167ccf9d29a3b010b', 'IT_dev.json': '825e158bfcb46c1001dc0062d43bdd8f4060c5214ec465aedd2c58d28da1b20a', 'label_vocab.json': 'd4d0175f792a5349ccc028ef3a1d220374169e999550441e84378dcf39e9ae45', 'CL_test.json': '33547ab29f5ae9afda36d92d779aa955676ae1d31df0ca479f09abfeeb48f248', 'IT_test.json': '3117dbe4a909bdec4628c200ed8ed2617a651d633296b41977b00f8b556dbc81'}
sys.path.insert(0,str(SRC))
import rr_data
from rr_data import (file_hash,read_json,write_json,find_input,prepare_rr_data,load_rr_documents,
    load_tokenizer,tokenize_documents,class_weights,load_own_checkpoint,split_chunks)
from architecture_models import ModelConfig,EncoderDecoderLM
from rr_encdec_model import EncoderDecoderRR,token_features
from rr_shared import trainable_state
from rr_encdec_preflight import run_core_checks,run_resource_probes,diagnostic_zip

rr_data.ROOT=ROOT
rr_data.INPUT_ROOTS=[Path('/kaggle/input')]
for name in ('preflight_report.json','core_checks.json','resource_probes.json','resource_probes.csv'):
    (RUN_DIR/name).unlink(missing_ok=True)
(RUN_DIR/'sources').mkdir(exist_ok=True)
for name in ['phase1_utils.py','a1_tokenizer.py','architecture_models.py','rr_data.py','rr_shared.py',
             'rr_check_utils.py','rr_encdec_model.py','rr_encdec_preflight.py']:
    shutil.copy2(SRC/name,RUN_DIR/'sources'/name)
environment={'python':sys.version,'torch':torch.__version__,
    'gpus':[{'name':torch.cuda.get_device_name(i),'total_gib':torch.cuda.get_device_properties(i).total_memory/2**30,
             'initial_free_gib':torch.cuda.mem_get_info(i)[0]/2**30} for i in range(2)],
    'versions':{n:importlib.metadata.version(n) for n in ('numpy','pandas','huggingface_hub')}}
write_json(RUN_DIR/'environment.json',environment)

DATA=prepare_rr_data(ROOT/'official_data/Phase_3_RR',EXPECTED_RR_HASHES,HF_REVISION,RR_DATA_INPUT)
labels,documents=load_rr_documents(DATA)
tokenizer_path=find_input('tokenizer_state.pkl',PREP_INPUT,hint='prep_002_mixed16k' if PREP_INPUT is None else None)
tokenizer,tokenizer_signature,manifest=load_tokenizer(tokenizer_path)
tokenizer_sha=file_hash(tokenizer_path)
if tokenizer_sha!='080c6d2ce49a5c9ccfdd707145ec2b49c8a3f3d2e676e30d9c6e93c948900d9c':
    raise ValueError('This is not the audited prep_002_mixed16k tokenizer.')
fingerprint=hashlib.sha256(json.dumps({'data_hashes':EXPECTED_RR_HASHES,'tokenizer_sha256':tokenizer_sha},
                                    sort_keys=True).encode()).hexdigest()
tokenize_documents(documents,tokenizer,fingerprint,ROOT/'token_cache')
weights,counts=class_weights(documents,labels,SETTINGS['class_weight_cap'])
phase1_checkpoint=find_input('best_model.pt',ENCDEC_INPUT,
    hint='encdec_002_mixed217m_stable' if ENCDEC_INPUT is None else None)
native_source=phase1_checkpoint.parent/'sources/architecture_models.py'
expected_architecture_sha='24f762e290b90a47ed54bd7314950ef2883d77b56d6f4fce0a02f39bf24a1e04'
if not native_source.exists():
    raise FileNotFoundError('Attach the complete Phase 1 encoder–decoder folder or backup including sources/architecture_models.py.')
if file_hash(native_source)!=expected_architecture_sha or file_hash(SRC/'architecture_models.py')!=expected_architecture_sha:
    raise ValueError('Original architecture source differs; inspect it before probing.')
checkpoint_sha=file_hash(phase1_checkpoint)
if checkpoint_sha!='2a2d34a8df48bce7b6ab8132ea011803e2d044df8eaacde110046653b85516a5':
    raise ValueError('Checkpoint bytes differ from the verified original encdec_002_mixed217m_stable checkpoint.')
saved=load_own_checkpoint(phase1_checkpoint)
if saved['tokenizer_signature']!=tokenizer_signature:
    raise ValueError('Phase 1 model and preparation tokenizers differ.')
cfg=ModelConfig(**saved['model_config'])
if cfg.architecture!='encoder_decoder' or cfg.vocab_size!=len(tokenizer.vocab):
    raise ValueError('Wrong architecture or vocabulary.')
lm=EncoderDecoderLM(cfg)
lm.load_state_dict(saved['model_state'],strict=True)
del saved
assert sum(p.numel() for p in lm.parameters())==217856000
input_record={'phase1_checkpoint_path':str(phase1_checkpoint),'phase1_checkpoint_sha256':checkpoint_sha,
    'architecture_source_sha256':expected_architecture_sha,'model_config':dict(cfg.__dict__),
    'tokenizer_sha256':tokenizer_sha,'tokenizer_signature':tokenizer_signature,
    'rr_data_revision':HF_REVISION,'rr_data_hashes':EXPECTED_RR_HASHES,
    'labels':labels,'training_counts':counts,'class_weights':weights.tolist(),
    'settings':SETTINGS,'proposed_training_only':PROPOSED_TRAINING,
    'source_sha256':{p.name:file_hash(p) for p in SRC.glob('*.py')}}
write_json(RUN_DIR/'input_record.json',input_record)
display(pd.DataFrame({'label':labels,'training_sentences':counts,'weight':weights.tolist()}))
print('Verified original Phase 1 checkpoint:',phase1_checkpoint)
print('RR sentences:',{s:sum(len(d['text']) for d in documents if d['split']==s) for s in ('train','dev','test')})

Loaded verified RR token cache.


,label,training_sentences,weight
0,Fact,4098,1.024469
1,Issue,107,5.000000
2,ArgumentPetitioner,1613,1.632930
3,ArgumentRespondent,1348,1.786240
4,Statute,1099,1.978271
5,Dissent,284,3.891576
6,PrecedentReliedUpon,1945,1.487048
7,PrecedentNotReliedUpon,569,2.749341
8,PrecedentOverruled,3,5.000000
9,RulingByLowerCourt,874,2.218345


Verified original Phase 1 checkpoint: /kaggle/input/datasets/ash9201kk/encdec-2/cs781_phase1/runs/encdec_002_mixed217m_stable/best_model.pt
RR sentences: {'train': 16845, 'dev': 2142, 'test': 2197}


## 4. Initialization and padding checks

Compare the task backbone's original features before and after inserting zero-effect adapters. Check attention masks, content pooling and right-padding invariance.


In [12]:
lm=lm.to('cuda:0').eval()
examples=[]
for domain in ('CL','IT'):
    content=next(ids for d in documents if d['split']=='train' and d['domain']==domain
                 for ids in d['token_ids'] if len(ids)>=5)
    examples.append([2]+content[:30]+[3])
lengths=torch.tensor([len(v) for v in examples],device='cuda:0')
ids=torch.zeros((len(examples),max(map(len,examples))),dtype=torch.long,device='cuda:0')
for i,v in enumerate(examples):
    ids[i,:len(v)]=torch.tensor(v,device='cuda:0')
with torch.no_grad():
    reference=token_features(lm,ids,lengths).cpu()
torch.manual_seed(SETTINGS['seed'])
base=EncoderDecoderRR(lm,len(labels),weights,SETTINGS).to('cuda:0').eval()
del lm
with torch.no_grad():
    adapted=token_features(base.lm,ids,lengths).cpu()
    torch.testing.assert_close(adapted,reference,rtol=1e-5,atol=1e-6)
    pooled=base.encode_chunks(ids,lengths).cpu()
    padded=torch.nn.functional.pad(ids,(0,7))
    padded_pooled=base.encode_chunks(padded,lengths).cpu()
    torch.testing.assert_close(pooled,padded_pooled,rtol=2e-4,atol=2e-5)
identity={'status':'passed','max_abs_difference':float((adapted-reference).abs().max()),
          'right_padding_invariance':'passed'}
parameter_record={'original_phase1_parameters':217856000,
    'lora_parameters':sum(p.numel() for n,p in base.named_parameters() if n.endswith(('.lora_A','.lora_B'))),
    'trainable_parameters':sum(p.numel() for p in base.parameters() if p.requires_grad),
    'adapter_targets':base.adapter_targets,'adapter_target_count':len(base.adapter_targets)}
write_json(RUN_DIR/'zero_init_and_padding.json',identity)
write_json(RUN_DIR/'parameter_record.json',parameter_record)
print('Identity:',identity)
print('LoRA parameters:',parameter_record['lora_parameters'],'| all trainable:',parameter_record['trainable_parameters'])
model=torch.nn.DataParallel(base,device_ids=[0,1])
del ids,lengths,reference,adapted,pooled,padded,padded_pooled
gc.collect()
torch.cuda.empty_cache()

Identity: {'status': 'passed', 'max_abs_difference': 0.0, 'right_padding_invariance': 'passed'}
LoRA parameters: 9830400 | all trainable: 10392784


## 5. Learning and resource checks

Check CRF numerics and two-GPU gradients, then run 40 temporary learning steps and probe heavy windows. The resource measurements guide the training configuration. The code restores the original parameters and RNG state after the probes.


In [13]:
report={'run_id':RUN_ID,'status':'running','identity':identity,'parameters':parameter_record,
        'proposed_training_only':PROPOSED_TRAINING}
try:
    report['core']=run_core_checks(model,documents,labels,weights,SETTINGS,RUN_DIR,learning_steps=LEARNING_STEPS)
    if report['core']['status']=='passed':
        report['resources']=run_resource_probes(model,documents,labels,weights,SETTINGS,RUN_DIR)
        report['status']=report['resources']['status']
    else:
        report['status']='failed'
        report['failure']='Core checks failed. Resource probes were not run.'
except Exception as error:
    report.update(status='failed',failure=str(error),traceback=traceback.format_exc())
finally:
    write_json(RUN_DIR/'preflight_report.json',report)
    diagnostics_zip=diagnostic_zip(RUN_DIR)
    print('PREFLIGHT STATUS:',report['status'])
    print('DIAGNOSTICS:',diagnostics_zip)

if report['status']=='passed':
    cases=report['resources']['cases']
    table=pd.DataFrame([{k:r.get(k) for k in ('window_sentences','overlap_sentences','chunk_microbatch',
        'status','median_heavy_update_seconds','peak_allocated_gib','peak_reserved_gib')} for r in cases])
    display(table)
    table.to_csv(RUN_DIR/'resource_probes.csv',index=False)
    print('Suggested configuration:',report['resources']['recommended_configuration'])
    print(json.dumps(report['resources']['timing'],indent=2))
    diagnostics_zip=diagnostic_zip(RUN_DIR)
    print('SEND THIS ZIP:',diagnostics_zip)
else:
    print('Send the diagnostics ZIP and the failure output. No full training has started.')

Temporary learning: 1 / 40 | loss 3.90573
Temporary learning: 10 / 40 | loss 1.44146
Temporary learning: 20 / 40 | loss 0.12881
Temporary learning: 30 / 40 | loss 0.01177
Temporary learning: 40 / 40 | loss 0.00408
Heavy probe: window 128, overlap 32, chunk microbatch 8
Result: passed | median update seconds: 11.90592881300006
Heavy probe: window 128, overlap 32, chunk microbatch 16
Result: passed | median update seconds: 11.252953802499974
PREFLIGHT STATUS: passed
DIAGNOSTICS: /kaggle/working/cs781_phase3/preflights/encdec_rr_lora_preflight_001_diagnostics.zip


,window_sentences,overlap_sentences,chunk_microbatch,status,median_heavy_update_seconds,peak_allocated_gib,peak_reserved_gib
0,128,32,8,passed,11.905929,"[3.331223487854004, 2.898357391357422]","[3.533203125, 3.08203125]"
1,128,32,16,passed,11.252954,"[3.709456443786621, 3.1400146484375]","[3.96484375, 3.46875]"


Suggested configuration: {'window_sentences': 128, 'overlap_sentences': 32, 'chunk_microbatch': 16, 'gradient_checkpointing': True}
{
  "status": "passed",
  "representative_update_seconds": [
    9.385879103999969,
    6.025881915000014,
    7.380554332000088
  ],
  "mean_update_seconds": 7.597438450333357,
  "train_windows": 197,
  "updates_per_epoch": 50,
  "dev_forward_seconds": 13.450929192999979,
  "estimated_training_seconds_per_epoch": 379.87192251666784,
  "estimated_full_train_eval_seconds": 107.0865547369816,
  "estimated_average_epoch_minutes": 7.15030616592214,
  "estimated_35_epoch_hours": 4.171011930121248,
  "notes": "Rough estimate from three sampled updates, all dev forward passes, and chunk-scaled train inference every three epochs. Excludes CRF decoding, metrics, checkpoint I/O and setup; early stopping may shorten the run."
}
SEND THIS ZIP: /kaggle/working/cs781_phase3/preflights/encdec_rr_lora_preflight_001_diagnostics.zip


## 6. Export the preflight proof

The preflight releases GPU memory and writes `encdec_rr_lora_preflight_001_diagnostics.zip`. Use it as `PREFLIGHT_INPUT` in the [training notebook](phase3_encoder_decoder_lora.ipynb) when starting in another session.


In [14]:
model.zero_grad(set_to_none=True)
del model,base
gc.collect()
torch.cuda.empty_cache()
print('Preflight model released. Download:',diagnostics_zip)

Preflight model released. Download: /kaggle/working/cs781_phase3/preflights/encdec_rr_lora_preflight_001_diagnostics.zip


## References

- Additive low-rank adapters. [LoRA](https://arxiv.org/abs/2106.09685)
- Context-aware legal sentence classification. [Semantic Segmentation of Legal Documents via Rhetorical Roles](https://aclanthology.org/2022.nllp-1.13/)
- Learned sequence scores and Viterbi decoding. [Neural Architectures for Named Entity Recognition](https://aclanthology.org/N16-1030/)
